# Benchmark: ImageFolder (local)

This notebook benchmarks a PyTorch `DataLoader` reading Food11 from a local filesystem path using `torchvision.datasets.ImageFolder`.

In the lab, the dataset is prepared by a Docker-based ETL pipeline and mounted into this Jupyter container at `/mnt/Food-11`.


In [ ]:
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

print('torch:', torch.__version__)


## Configuration


In [ ]:
DATA_ROOT = os.environ.get('FOOD11_DATA_DIR', '/mnt/Food-11')
SPLIT = 'training'

BATCH_SIZE = 64
NUM_WORKERS = 8

WARMUP_BATCHES = 10
MEASURE_BATCHES = 200


print('DATA_ROOT:', DATA_ROOT)
print('SPLIT:', SPLIT)
print('BATCH_SIZE:', BATCH_SIZE)
print('NUM_WORKERS:', NUM_WORKERS)
print('WARMUP_BATCHES:', WARMUP_BATCHES)
print('MEASURE_BATCHES:', MEASURE_BATCHES)



## Dataset


In [ ]:
split_dir = os.path.join(DATA_ROOT, SPLIT)
if not os.path.isdir(split_dir):
    raise FileNotFoundError(f'Missing split directory: {split_dir} (is the dataset mounted?)')

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

dataset = datasets.ImageFolder(root=split_dir, transform=transform)

print('num_samples:', len(dataset))


## DataLoader


In [ ]:
num_workers = NUM_WORKERS
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    drop_last=False,
    prefetch_factor=2,
    persistent_workers=True,
)


## Run benchmark


In [ ]:
it = iter(loader)

for _ in range(WARMUP_BATCHES):
    try:
        _ = next(it)
    except StopIteration:
        break

num_batches = 0
num_items = 0
t_start = time.perf_counter()
for _ in range(MEASURE_BATCHES):
    try:
        x, y = next(it)
    except StopIteration:
        break
    num_batches += 1
    num_items += int(y.shape[0])
t_end = time.perf_counter()

wall_s = t_end - t_start
imgs_per_s = (num_items / wall_s) if wall_s > 0 else float('nan')
batches_per_s = (num_batches / wall_s) if wall_s > 0 else float('nan')
avg_batch_s = (wall_s / num_batches) if num_batches > 0 else None

result = {
    'num_workers': num_workers,
    'batch_size': BATCH_SIZE,
    'measured_batches': num_batches,
    'measured_items': num_items,
    'wall_s': wall_s,
    'imgs_per_s': imgs_per_s,
    'batches_per_s': batches_per_s,
    'avg_batch_s': avg_batch_s,
}

result


## Print results


In [ ]:
print('split:', SPLIT)
print('batch_size:', BATCH_SIZE)
print('warmup_batches:', WARMUP_BATCHES, 'measure_batches:', MEASURE_BATCHES)
print()

avg_batch_s = result['avg_batch_s']
avg_batch_s_str = 'nan' if avg_batch_s is None else f"{avg_batch_s:.4f}"
print(
    'workers=', result['num_workers'],
    'imgs/s=', f"{result['imgs_per_s']:.2f}",
    'batches/s=', f"{result['batches_per_s']:.2f}",
    'avg_batch_s=', avg_batch_s_str,
)


## Save results


In [ ]:
out_dir = Path('results')
out_dir.mkdir(parents=True, exist_ok=True)

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
out_path = out_dir / f'imagefolder_local_{stamp}.json'
payload = {
    'benchmark': 'imagefolder_local',
    'timestamp_utc': stamp,
    'data_root': DATA_ROOT,
    'split': SPLIT,
    'batch_size': BATCH_SIZE,
    'warmup_batches': WARMUP_BATCHES,
    'measure_batches': MEASURE_BATCHES,
    'result': result,
}
out_path.write_text(json.dumps(payload, indent=2))
print('Wrote:', out_path)
